# A Convolutional Network with Keras

In notebook 01 you computed convolutions and pooling with NumPy and chose the filters yourself. In this notebook Keras does the computing and the network learns its own filters. You build a classic convolutional network, LeNet-5, for the MNIST digits, compare it with a dense network and look at what its first layer has learnt.

## Learning Objectives

By the end of this notebook, you should be able to:

- Prepare image data for a Keras CNN: the channel axis, the scaling and a validation set.
- Calculate the output shape and the parameters of each layer of a CNN before you build it, and check them against `model.summary()`.
- Build a CNN with `Conv2D`, `MaxPooling2D`, `Flatten` and `Dense` layers.
- Compare a CNN with a dense network by their validation accuracy and their number of parameters.
- Plot the learnt filters and the feature maps of a convolutional layer.
- Evaluate the chosen model on the test set and identify its most frequent mistakes in a confusion matrix.

## Setup

In [ ]:
import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

# One plot style for the whole notebook
plt.rcParams.update(
    {"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3}
)

## 1. Load and Prepare the Data

[`keras.datasets.mnist.load_data()`](https://keras.io/api/datasets/mnist/#load_data-function) returns 60,000 training and 10,000 test images of handwritten digits, each 28 × 28 pixels with values from 0 to 255, and their labels from 0 to 9. It downloads the data on the first run.

In [ ]:
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.mnist.load_data()
print("Training images:", X_train_full.shape, " test images:", X_test.shape)

fig, axes = plt.subplots(1, 10, figsize=(14, 1.8))
for ax, image, label in zip(axes, X_train_full, y_train_full):
    ax.imshow(image, cmap="gray")
    ax.set_title(label)
    ax.axis("off")
plt.show()

Three preparation steps follow.

1. **A validation set.** Keep 10,000 of the training images aside to compare the models; the test set is used once, at the end. Use [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) with `stratify` so every digit keeps its share.
2. **A channel axis.** A Keras convolutional layer expects every image as a 3D array of height, width and channels, so a batch has the shape `(images, 28, 28, 1)`. Greyscale images have one channel; colour images would have three. `X[..., np.newaxis]` adds the axis at the end.
3. **Scaling.** The pixel values are divided by 255 inside the model, with a [`Rescaling`](https://keras.io/api/layers/preprocessing_layers/image_preprocessing/rescaling/) layer, so the model accepts raw images and the scaling is saved with it.

The labels stay integers from 0 to 9, so the models use the `sparse_categorical_crossentropy` loss, as in the multiclass classification networks you built before.

In [ ]:
# TODO: split X_train_full and y_train_full into a training set and a validation set of 10,000 images,
# with stratify=y_train_full and random_state=42
X_train, X_valid, y_train, y_valid = ...

# TODO: add a channel axis at the end of X_train, X_valid and X_test
X_train = ...
X_valid = ...
X_test = ...
print("Training:", X_train.shape, " validation:", X_valid.shape, " test:", X_test.shape)

## 2. A Keras Convolutional Layer Computes the Same Convolution

A [`Conv2D`](https://keras.io/api/layers/convolution_layers/convolution2d/) layer performs the convolution you wrote in notebook 01. To check, the cell below creates a layer with a single 3 × 3 filter and no bias, creates its weights with `build()`, replaces them with the vertical Sobel filter with [`set_weights()`](https://keras.io/api/layers/base_layer/) and applies the layer to the first training image. Keras stores the filters of a layer in an array of shape `(height, width, input channels, filters)`, here `(3, 3, 1, 1)`.

In [ ]:
def convolve(image, kernel):
    """The NumPy convolution of notebook 01 (no padding, stride 1)."""
    n, f = image.shape[0], kernel.shape[0]
    feature_map = np.zeros((n - f + 1, n - f + 1))
    for row in range(n - f + 1):
        for col in range(n - f + 1):
            feature_map[row, col] = np.sum(image[row:row + f, col:col + f] * kernel)
    return feature_map


sobel_vertical = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype="float32")
sobel_layer = keras.layers.Conv2D(filters=1, kernel_size=3, use_bias=False)
sobel_layer.build((None, 28, 28, 1))  # create the weights for one-channel 28 x 28 images
sobel_layer.set_weights([sobel_vertical.reshape(3, 3, 1, 1)])

image = X_train[0] / 255  # shape (28, 28, 1)
keras_map = np.asarray(sobel_layer(image[np.newaxis]))[0, :, :, 0]  # one image in, one feature map out
numpy_map = convolve(image[:, :, 0], sobel_vertical)
print("Keras feature map:", keras_map.shape)
print(f"Largest difference to the NumPy feature map: {np.abs(keras_map - numpy_map).max():.1e}")
print("Same up to rounding:", np.allclose(keras_map, numpy_map, atol=1e-6))

The two feature maps agree up to a difference of a few times $10^{-7}$. Keras computes with 32-bit numbers, NumPy here with 64-bit numbers, so the last digits are rounded differently; `atol=1e-6` allows for this. In a CNN you do not set the filters yourself: they start random, and `fit()` learns them with backpropagation.

## 3. Baseline: a Dense Network

A CNN is worth its extra layers only if it does better than a simpler model. The baseline is a dense network with one hidden layer of 128 neurons. [`Flatten`](https://keras.io/api/layers/reshaping_layers/flatten/) turns every 28 × 28 × 1 image into a list of 784 numbers, which is exactly the loss of neighbourhood information notebook 01 described.

Two helper functions compile and train a model with the same settings every time, and plot its learning curves.

In [ ]:
def compile_and_fit(model, epochs=10):
    """Compile a classifier with Adam and train it; return the learning curves as a DataFrame."""
    model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        metrics=["accuracy"],
    )
    history = model.fit(
        X_train, y_train, epochs=epochs, batch_size=128, validation_data=(X_valid, y_valid), verbose=0
    )
    return pd.DataFrame(history.history)


def plot_curves(logs, title):
    """Plot the training and validation loss and accuracy of one run."""
    fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4))
    logs[["loss", "val_loss"]].plot(ax=ax_loss, xlabel="Epoch", title=f"{title}: loss")
    logs[["accuracy", "val_accuracy"]].plot(ax=ax_acc, xlabel="Epoch", title=f"{title}: accuracy")
    plt.show()


results = {}

In [ ]:
keras.utils.clear_session()
keras.utils.set_random_seed(42)
dense_model = keras.Sequential(
    [
        keras.Input(shape=(28, 28, 1)),
        keras.layers.Rescaling(1 / 255),
        keras.layers.Flatten(),
        keras.layers.Dense(units=128, activation="relu"),
        keras.layers.Dense(units=10, activation="softmax"),
    ]
)
dense_logs = compile_and_fit(dense_model)
plot_curves(dense_logs, "Dense network")
results["Dense network"] = {
    "parameters": dense_model.count_params(),
    "validation accuracy": dense_model.evaluate(X_valid, y_valid, verbose=0)[1],
}
pd.DataFrame(results).T

The dense network has 101,770 parameters, almost all of them in the hidden layer ($784 \cdot 128 + 128 = 100{,}480$). After 10 epochs it reaches a validation accuracy of 0.975, so it misclassifies about 2.5 % of the validation digits. Its training accuracy keeps rising to 0.99 while the validation accuracy levels off, and the gap between the loss curves grows: a first sign of overfitting.

## 4. A First CNN: One Convolutional Block

The smallest useful CNN has one **convolutional block**, a convolutional layer followed by a pooling layer, and a dense output layer:

`Rescaling` → `Conv2D` with 8 filters of 3 × 3 → `MaxPooling2D` 2 × 2 → `Flatten` → `Dense` with 10 softmax neurons.

**Your turn:** before you run the next cell, use the formulas of notebook 01 to predict the output shape and the number of parameters of every layer. Then compare with `model.summary()`.

<details>
<summary>Show the answer</summary>

| Layer | Output shape | Parameters |
|---|---|---|
| `Conv2D`, 8 filters 3 × 3, valid | 26 × 26 × 8 ($28 - 3 + 1 = 26$) | $(3 \cdot 3 \cdot 1 + 1) \cdot 8 = 80$ |
| `MaxPooling2D` 2 × 2 | 13 × 13 × 8 | 0 |
| `Flatten` | $13 \cdot 13 \cdot 8 = 1{,}352$ | 0 |
| `Dense`, 10 | 10 | $1{,}352 \cdot 10 + 10 = 13{,}530$ |

In total 13,610 parameters, about an eighth of the dense baseline's.

</details>

In [ ]:
keras.utils.clear_session()
keras.utils.set_random_seed(42)
first_cnn = keras.Sequential(
    [
        keras.Input(shape=(28, 28, 1)),
        keras.layers.Rescaling(1 / 255),
        keras.layers.Conv2D(filters=8, kernel_size=3, activation="relu"),
        keras.layers.MaxPooling2D(pool_size=2),
        keras.layers.Flatten(),
        keras.layers.Dense(units=10, activation="softmax"),
    ]
)
first_cnn.summary()

The cell below trains the first CNN with the same settings as the dense baseline.

In [ ]:
first_logs = compile_and_fit(first_cnn)
plot_curves(first_logs, "First CNN")
results["First CNN"] = {
    "parameters": first_cnn.count_params(),
    "validation accuracy": first_cnn.evaluate(X_valid, y_valid, verbose=0)[1],
}
pd.DataFrame(results).T

With only 13,610 parameters, an eighth of the dense baseline's 101,770, the first CNN reaches a validation accuracy of 0.970, close to the dense network's 0.975. Its learning curves are still rising at epoch 10, and the training and validation accuracy stay close together. A single convolutional block is already almost as good as a dense network of eight times its size; LeNet-5 shows what a second block adds.

## 5. LeNet-5

**LeNet-5** (LeCun and colleagues, 1998) was one of the first CNNs used in practice; banks used it to read the handwritten amounts on cheques. It stacks two convolutional blocks, a **convolutional stack** that extracts features from the image, and ends with dense layers that classify. The network below is a modern version of LeNet-5: it uses ReLU, max pooling and a softmax output, where the original used other activations, a trainable average pooling and a different output layer.

<img src="assets/lenet.png" width="1000">

The original network takes 32 × 32 images. For 28 × 28 digits, the first convolutional layer uses `padding="same"`, so its output keeps the size of 28 × 28. [`MaxPooling2D`](https://keras.io/api/layers/pooling_layers/max_pooling2d/) with `pool_size=2` halves the height and the width, and the second convolutional layer has no padding (`padding="valid"`, the default).

**Your turn:** before you run the next cell, predict the output shape of every layer from the figure and the formulas of notebook 01, and the number of parameters of the two convolutional layers. The table after the summary gives the answers.

In [ ]:
keras.utils.clear_session()
keras.utils.set_random_seed(42)
lenet = keras.Sequential(
    [
        keras.Input(shape=(28, 28, 1)),
        keras.layers.Rescaling(1 / 255),
        keras.layers.Conv2D(filters=6, kernel_size=5, padding="same", activation="relu"),
        keras.layers.MaxPooling2D(pool_size=2),
        keras.layers.Conv2D(filters=16, kernel_size=5, activation="relu"),
        keras.layers.MaxPooling2D(pool_size=2),
        keras.layers.Flatten(),
        keras.layers.Dense(units=120, activation="relu"),
        keras.layers.Dense(units=84, activation="relu"),
        keras.layers.Dense(units=10, activation="softmax"),
    ]
)
lenet.summary()

Every shape and parameter count in the summary follows from the formulas of notebook 01:

| Layer | Output shape | How it is calculated | Parameters |
|---|---|---|---|
| `Conv2D`, 6 filters 5 × 5, same | 28 × 28 × 6 | padding keeps 28; one channel per filter | $(5 \cdot 5 \cdot 1 + 1) \cdot 6 = 156$ |
| `MaxPooling2D` | 14 × 14 × 6 | 28 / 2 | 0 |
| `Conv2D`, 16 filters 5 × 5, valid | 10 × 10 × 16 | $14 - 5 + 1 = 10$ | $(5 \cdot 5 \cdot 6 + 1) \cdot 16 = 2{,}416$ |
| `MaxPooling2D` | 5 × 5 × 16 | 10 / 2 | 0 |
| `Flatten` | 400 | $5 \cdot 5 \cdot 16$ | 0 |
| `Dense`, 120 | 120 | | $400 \cdot 120 + 120 = 48{,}120$ |
| `Dense`, 84 | 84 | | $120 \cdot 84 + 84 = 10{,}164$ |
| `Dense`, 10 | 10 | | $84 \cdot 10 + 10 = 850$ |

The two convolutional layers have only 2,572 parameters together. Most parameters sit in the first dense layer, which connects the 400 extracted features to 120 neurons. In total LeNet-5 has 61,706 parameters, fewer than the 101,770 of the dense baseline.

### Why Dense Layers on Top of the Convolutional Layers?

The convolutional stack and the dense layers do different jobs:

- **The convolutional layers find features and where they are.** Their output is a box of feature maps, 5 × 5 × 16 in LeNet-5. Each value describes one small region of the image: how strongly one filter responded there. No single value says which digit the image shows.
- **The dense layers combine the features of the whole image into a decision.** A 7 is a horizontal line at the top *and* a diagonal line below it; a 1 has the diagonal line but not the horizontal one. To tell them apart, the network has to weigh features from different regions together. A dense layer connects every value of the box to every one of its neurons, so it can learn exactly such combinations. It works like the feed-forward networks you built for tabular data, but its inputs are learnt image features instead of raw pixels.
- **The output layer has to give one probability per class.** A dense layer with 10 softmax neurons produces exactly that. `Flatten` turns the 5 × 5 × 16 box into a list of 400 values first, because a dense layer expects a flat list as input.

The dense layers come only at the end because by then the convolutions and pooling have reduced the 784 pixels to 400 features that are robust to small shifts. A dense layer directly on the pixels would treat neighbouring pixels as unrelated, as the dense baseline does, and its weights would be tied to fixed pixel positions. Modern networks often replace `Flatten` with [`GlobalAveragePooling2D`](https://keras.io/api/layers/pooling_layers/global_average_pooling2d/), which averages each feature map to a single number; you will meet it in pretrained networks.

In [ ]:
lenet_logs = compile_and_fit(lenet)
plot_curves(lenet_logs, "LeNet-5")
results["LeNet-5"] = {
    "parameters": lenet.count_params(),
    "validation accuracy": lenet.evaluate(X_valid, y_valid, verbose=0)[1],
}
pd.DataFrame(results).T

LeNet-5 reaches a validation accuracy of 0.987 with 61,706 parameters, clearly better than the first CNN's 0.970. It makes about half as many mistakes as the dense network (1.3 % instead of 2.5 % of the validation digits) with 40 % fewer parameters. Its validation accuracy is already 0.98 after two epochs, and the validation loss stays flat at about 0.05 from epoch 3 onwards while the training loss keeps falling, so more epochs would mainly improve the fit to the training set. The difference comes from the convolutional layers: the same small filters are applied at every position, so the network recognises a line wherever it appears in the image, while every weight of the dense network belongs to one fixed pixel.

## 6. What the First Layer Has Learnt

The six filters of the first convolutional layer are its weights, an array of shape `(5, 5, 1, 6)`. The cell below plots each one as a 5 × 5 image: red for positive and blue for negative weights.

In [ ]:
filters = lenet.layers[1].get_weights()[0]  # layers[0] is Rescaling, layers[1] the first Conv2D
print("Filter weights:", filters.shape)

fig, axes = plt.subplots(1, 6, figsize=(13, 2.6))
limit = np.abs(filters).max()
for i, ax in enumerate(axes):
    ax.imshow(filters[:, :, 0, i], cmap="RdBu_r", vmin=-limit, vmax=limit)
    ax.set_title(f"Filter {i}")
    ax.axis("off")
plt.show()

To see what the filters respond to, apply them to one image. A Keras model can be cut after any layer: [`keras.Model`](https://keras.io/api/models/model/) with the inputs of LeNet-5 and the output of its first convolutional layer returns the six feature maps of that layer.

In [ ]:
first_layer = keras.Model(inputs=lenet.inputs, outputs=lenet.layers[1].output)
feature_maps = first_layer.predict(X_valid[:1], verbose=0)[0]  # shape (28, 28, 6)
print("Feature maps:", feature_maps.shape)

fig, axes = plt.subplots(1, 7, figsize=(15, 2.6))
axes[0].imshow(X_valid[0, :, :, 0], cmap="gray")
axes[0].set_title(f"Digit {y_valid[0]}")
for i, ax in enumerate(axes[1:]):
    ax.imshow(feature_maps[:, :, i], cmap="gray")
    ax.set_title(f"Feature map {i}")
for ax in axes:
    ax.axis("off")
plt.show()

The first validation image is a 7. Each filter responds to a different part of it; in the reference run of this notebook: Feature maps 3 and 4 respond most strongly to the horizontal line at the top, feature map 1 mainly to the right edge of the diagonal line, and feature map 0 to the whole digit in a blurred form. These are the edge and line detectors of notebook 01, but the network has found them by itself, because they help to tell the digits apart. The filters in the plot above are harder to read than the Sobel filter, but some show the same structure: filter 1, for example, is positive on its left and negative on its right, like a vertical-edge filter.

## Try It: Design Your Own CNN

Change the architecture of LeNet-5 and compare your network with it on the **validation** set. Aim to beat LeNet-5's validation accuracy of 0.987, or to match it with fewer than its 61,706 parameters. Ideas: more or fewer filters, `padding="same"`, a third convolutional block, or a [`Dropout`](https://keras.io/api/layers/regularization_layers/dropout/) layer before the dense layers. Predict the shapes before you look at `model.summary()`. The test set is kept for the final check in section 7.

In [ ]:
keras.utils.clear_session()
keras.utils.set_random_seed(42)
my_cnn = keras.Sequential(
    [
        keras.Input(shape=(28, 28, 1)),
        keras.layers.Rescaling(1 / 255),
        # TODO: one or more convolutional blocks: Conv2D with a ReLU activation, followed by MaxPooling2D
        ...,
        keras.layers.Flatten(),
        # TODO: one or more Dense hidden layers, optionally with Dropout
        ...,
        keras.layers.Dense(units=10, activation="softmax"),
    ]
)
my_cnn.summary()

In [ ]:
# TODO: train my_cnn with compile_and_fit and plot its learning curves
my_logs = ...

valid_accuracy = my_cnn.evaluate(X_valid, y_valid, verbose=0)[1]
print(f"Validation accuracy {valid_accuracy:.4f} with {my_cnn.count_params():,} parameters "
      f"(LeNet-5: {results['LeNet-5']['validation accuracy']:.4f} with {lenet.count_params():,})")

# TODO: plot the six first feature maps of the first convolutional layer of my_cnn for X_valid[:1],
# as in section 6 (layers[1] is the first Conv2D)
...

## 7. Final Check on the Test Set

The models were compared on the validation set, and LeNet-5 is the chosen model. The test set gives an unbiased estimate of its accuracy and is used once, here. The dense network's test accuracy is shown for reference only: the choice has already been made.

In [ ]:
for name, model in {"LeNet-5": lenet, "Dense network": dense_model}.items():
    results[name]["test accuracy"] = model.evaluate(X_test, y_test, verbose=0)[1]
pd.DataFrame(results).T.loc[["LeNet-5", "Dense network"]]

A [confusion matrix](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html) shows which digits LeNet-5 still confuses. Each row is the true digit and each column the prediction; the diagonal holds the correct predictions, so the mistakes are the numbers off the diagonal.

In [ ]:
y_pred = lenet.predict(X_test, verbose=0).argmax(axis=1)

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, colorbar=False, ax=ax)
ax.grid(False)
plt.show()

wrong = np.flatnonzero(y_pred != y_test)
print(f"{len(wrong)} of {len(y_test)} test images are misclassified")
fig, axes = plt.subplots(1, 8, figsize=(14, 2.2))
for ax, i in zip(axes, wrong):
    ax.imshow(X_test[i, :, :, 0], cmap="gray")
    ax.set_title(f"true {y_test[i]}, pred {y_pred[i]}", fontsize=9)
    ax.axis("off")
plt.show()

On the test set, the chosen model, LeNet-5, reaches an accuracy of 0.989, close to its validation accuracy of 0.987. The dense network reaches 0.976, shown for reference. LeNet-5 misclassifies 108 of the 10,000 test digits. Its most frequent mistake is a 9 predicted as a 4 (10 times), followed by a 5 predicted as a 3 and a 7 predicted as a 2 (6 times each). The misclassified examples of the reference run show why: many of them are written unusually, such as a 4 with an open top or a 3 that is drawn almost flat, and some would be hard to read for a person as well.

## Summary

- A Keras CNN expects images with a channel axis, `(images, height, width, channels)`. A `Rescaling` layer scales the pixels inside the model.
- `Conv2D` computes the same convolution as the NumPy code of notebook 01, but learns its filters with backpropagation. `MaxPooling2D` halves the height and the width, and `Flatten` turns the last feature maps into the input of the dense layers.
- The shapes and parameters in `model.summary()` follow from the output-size formula and from $(f \cdot f \cdot c + 1) \cdot k$ for each convolutional layer.
- The convolutional layers find local features and where they are; the dense layers on top combine the features of the whole image into the class probabilities.
- On MNIST, LeNet-5 makes fewer mistakes than a dense network with more parameters, because its filters use the neighbourhood of every pixel.
- The filters and feature maps of the first layer show that it has learnt to detect edges and lines.

Next, apply what you have learnt in [03 - Exercise: Classify Clothing Images with a CNN](03_exercise_fashion_mnist.ipynb).

## References

- LeCun, Y., Bottou, L., Bengio, Y. and Haffner, P. (1998). [Gradient-based learning applied to document recognition](http://yann.lecun.com/exdb/publis/pdf/lecun-01a.pdf). *Proceedings of the IEEE*, 86(11), 2278-2324: the LeNet-5 paper.
- Chollet, F. and Watson, M. (2025). *Deep Learning with Python*, 3rd edition, chapter 8: Image classification. [Free to read online](https://deeplearningwithpython.io/).
- TensorFlow tutorial: [Convolutional Neural Network (CNN)](https://www.tensorflow.org/tutorials/images/cnn).
- Harley, A. W. (2015). [An Interactive Node-Link Visualization of Convolutional Neural Networks](https://adamharley.com/nn_vis/cnn/2d.html): draw a digit and watch every layer of a CNN respond.
- Keras API: [`Conv2D`](https://keras.io/api/layers/convolution_layers/convolution2d/), [`MaxPooling2D`](https://keras.io/api/layers/pooling_layers/max_pooling2d/), [`Flatten`](https://keras.io/api/layers/reshaping_layers/flatten/) and [`Rescaling`](https://keras.io/api/layers/preprocessing_layers/image_preprocessing/rescaling/).
- The structure follows the convolutional neural networks notebook of the Spiced Academy deep learning course.